# MedGemma 4B ESI Fine-Tuning

QLoRA fine-tuning and held-out evaluation on authorized MIMIC-IV-ED triage
data. This notebook is for research only.

## Configuration

Choose one profile before running the notebook:

```python
PROFILE = "smoke"         # 20/10/10 records, 1 update
PROFILE = "larger_smoke"  # 500/100/100 records, 20 updates
PROFILE = "mimic_smoke"   # 500/100/100 records, 60 updates
PROFILE = "full"          # full training split, 1 epoch
```

Training requires CUDA. Evaluation uses CUDA when available and otherwise
runs on CPU.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json
import os
import platform
import re
import sys
import time

import pandas as pd
import torch
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "prepare_training_data.py").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the oss_model_clinical_triage_demo repository.")


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
PROJECT_ROOT

In [ ]:
MODEL_ID = "google/medgemma-1.5-4b-it"
LOCAL_FILES_ONLY = True

PROFILE = "mimic_smoke"  # smoke, larger_smoke, mimic_smoke, or full
DEVICE_KIND = "cuda" if torch.cuda.is_available() else "cpu"
REVIEW_CONFIDENCE_THRESHOLD = 0.60
FORCE_GPU_ONLY_LOAD = True

RUN_BASE_EVALUATION = False
RUN_TRAINING = False
RUN_ADAPTER_EVALUATION = False

PROFILES = {
    "smoke": {
        "data_dir": "data/finetune_mimic_smoke",
        "output_dir": "outputs/notebook_smoke",
        "train_limit": 20,
        "validation_limit": 10,
        "test_limit": 10,
        "max_steps": 1,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 1,
        "max_length": 256,
        "learning_rate": 2e-4,
        "lora_r": 4,
        "lora_alpha": 8,
        "eval_strategy": "no",
        "save_strategy": "steps",
        "save_steps": 1,
    },
    "larger_smoke": {
        "data_dir": "data/finetune_mimic_smoke",
        "output_dir": "outputs/notebook_larger_smoke",
        "train_limit": 500,
        "validation_limit": 100,
        "test_limit": 100,
        "max_steps": 20,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "max_length": 512,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "eval_strategy": "steps",
        "save_strategy": "steps",
        "save_steps": 10,
    },
    "mimic_smoke": {
        "data_dir": "data/finetune_mimic_smoke",
        "output_dir": "outputs/notebook_mimic_smoke",
        "train_limit": 500,
        "validation_limit": 100,
        "test_limit": 100,
        "max_steps": 60,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "max_length": 512,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "eval_strategy": "steps",
        "save_strategy": "steps",
        "save_steps": 20,
    },
    "full": {
        "data_dir": "data/finetune_mimic",
        "output_dir": "outputs/notebook_mimic_full",
        "train_limit": None,
        "validation_limit": 500,
        "test_limit": 1000,
        "max_steps": -1,
        "epochs": 1,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "max_length": 512,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "eval_strategy": "epoch",
        "save_strategy": "epoch",
        "save_steps": 500,
    },
}

if PROFILE not in PROFILES:
    raise ValueError(f"Unknown profile: {PROFILE}")

CONFIG = dict(PROFILES[PROFILE])
CONFIG["device_kind"] = DEVICE_KIND
CONFIG["review_confidence_threshold"] = REVIEW_CONFIDENCE_THRESHOLD
CONFIG["force_gpu_only_load"] = FORCE_GPU_ONLY_LOAD
DATA_DIR = PROJECT_ROOT / CONFIG["data_dir"]
OUTPUT_DIR = PROJECT_ROOT / CONFIG["output_dir"]
ADAPTER_DIR = OUTPUT_DIR / "adapter"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.Series(CONFIG, name=PROFILE)


## Data

Load the prepared JSONL splits and confirm that patients do not cross splits.

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def apply_limit(records: list[dict], limit: int | None) -> list[dict]:
    return records if limit is None else records[:limit]


train_records = apply_limit(load_jsonl(DATA_DIR / "train.jsonl"), CONFIG["train_limit"])
validation_records = apply_limit(
    load_jsonl(DATA_DIR / "validation.jsonl"),
    CONFIG["validation_limit"],
)
test_records = apply_limit(load_jsonl(DATA_DIR / "test.jsonl"), CONFIG["test_limit"])

subject_sets = {
    "train": {record["subject_id"] for record in train_records},
    "validation": {record["subject_id"] for record in validation_records},
    "test": {record["subject_id"] for record in test_records},
}
assert subject_sets["train"].isdisjoint(subject_sets["validation"])
assert subject_sets["train"].isdisjoint(subject_sets["test"])
assert subject_sets["validation"].isdisjoint(subject_sets["test"])

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train_records), len(validation_records), len(test_records)],
        "subjects": [len(subject_sets[name]) for name in ("train", "validation", "test")],
    }
)
display(split_summary)
print("No subject overlap.")

## Metrics

Higher ESI numbers indicate lower urgency:

- prediction above the label: under-triage
- prediction below the label: over-triage
- ESI 1-2 predicted as ESI 4-5: severe under-triage

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, recall_score
from tqdm.auto import tqdm


def parse_esi_prediction(text: str):
    match = re.search(r"(uncertain|[1-5])", str(text), flags=re.IGNORECASE)
    if not match:
        return None
    value = match.group(1).lower()
    return "uncertain" if value == "uncertain" else int(value)


def needs_human_review(prediction, confidence: float | None) -> tuple[bool, str]:
    if prediction is None:
        return True, "invalid_output"
    if prediction == "uncertain":
        return True, "model_uncertain"
    if confidence is not None and confidence < REVIEW_CONFIDENCE_THRESHOLD:
        return True, "low_confidence"
    return False, ""


def test_fingerprint(records: list[dict]) -> str:
    keys = [
        f"{record['subject_id']}:{record['stay_id']}:{record['label']}"
        for record in records
    ]
    return hashlib.sha256("|".join(keys).encode("utf-8")).hexdigest()


def compute_clinical_metrics(predictions: list[dict]) -> dict:
    total = len(predictions)
    actual = [row["actual"] for row in predictions]
    predicted = [row["prediction"] for row in predictions]
    numeric_prediction = [value if isinstance(value, int) else 0 for value in predicted]
    valid_mask = [isinstance(value, int) and 1 <= value <= 5 for value in predicted]

    correct = sum(
        prediction == label
        for prediction, label in zip(predicted, actual)
    )
    under_triage = sum(
        isinstance(prediction, int) and prediction > label
        for prediction, label in zip(predicted, actual)
    )
    over_triage = sum(
        isinstance(prediction, int) and prediction < label
        for prediction, label in zip(predicted, actual)
    )
    severe_under_triage = sum(
        label in (1, 2) and isinstance(prediction, int) and prediction in (4, 5)
        for prediction, label in zip(predicted, actual)
    )
    uncertain = sum(value == "uncertain" for value in predicted)
    invalid = sum(value is None for value in predicted)
    human_review = sum(
        bool(row.get("needs_human_review", row["prediction"] in {None, "uncertain"}))
        for row in predictions
    )
    low_confidence_review = sum(
        row.get("review_reason") == "low_confidence"
        for row in predictions
    )
    confidence_values = [
        row.get("prediction_confidence")
        for row in predictions
        if row.get("prediction_confidence") is not None
    ]

    recalls = recall_score(
        actual,
        numeric_prediction,
        labels=[1, 2, 3, 4, 5],
        average=None,
        zero_division=0,
    )
    matrix = confusion_matrix(actual, numeric_prediction, labels=[1, 2, 3, 4, 5])

    return {
        "examples": total,
        "valid_predictions": int(sum(valid_mask)),
        "coverage": sum(valid_mask) / total if total else 0.0,
        "accuracy": correct / total if total else 0.0,
        "macro_f1": f1_score(
            actual,
            numeric_prediction,
            labels=[1, 2, 3, 4, 5],
            average="macro",
            zero_division=0,
        ),
        "under_triage_rate": under_triage / total if total else 0.0,
        "over_triage_rate": over_triage / total if total else 0.0,
        "severe_under_triage_rate": severe_under_triage / total if total else 0.0,
        "uncertainty_rate": uncertain / total if total else 0.0,
        "invalid_output_rate": invalid / total if total else 0.0,
        "human_review_rate": human_review / total if total else 0.0,
        "low_confidence_review_rate": low_confidence_review / total if total else 0.0,
        "average_prediction_confidence": (
            sum(confidence_values) / len(confidence_values)
            if confidence_values
            else None
        ),
        "review_confidence_threshold": REVIEW_CONFIDENCE_THRESHOLD,
        "recall_by_esi": {
            str(label): float(value)
            for label, value in zip([1, 2, 3, 4, 5], recalls)
        },
        "confusion_matrix_labels_1_to_5": matrix.tolist(),
    }


def save_evaluation(name: str, predictions: list[dict]) -> dict:
    result = {
        "model_id": MODEL_ID,
        "profile": PROFILE,
        "test_fingerprint": test_fingerprint(test_records),
        "metrics": compute_clinical_metrics(predictions),
        "predictions": predictions,
    }
    json_path = OUTPUT_DIR / f"{name}.json"
    csv_path = OUTPUT_DIR / f"{name}.csv"
    json_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    pd.DataFrame(predictions).to_csv(csv_path, index=False)
    print(f"Saved {json_path}")
    return result


def display_metrics(result: dict):
    metrics = result["metrics"]
    scalar_metrics = {
        key: value
        for key, value in metrics.items()
        if key not in {"recall_by_esi", "confusion_matrix_labels_1_to_5"}
    }
    display(pd.DataFrame([scalar_metrics]))
    display(
        pd.DataFrame(
            metrics["confusion_matrix_labels_1_to_5"],
            index=[f"actual_{i}" for i in range(1, 6)],
            columns=[f"predicted_{i}" for i in range(1, 6)],
        )
    )
    display(
        pd.DataFrame.from_dict(
            metrics["recall_by_esi"],
            orient="index",
            columns=["recall"],
        )
    )


In [ ]:
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
)


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def release_model_memory():
    for name in (
        "base_model",
        "base_processor",
        "adapter_model",
        "adapter_processor",
        "trainer",
        "model",
        "processor",
    ):
        globals().pop(name, None)
    cleanup_cuda()


def print_gpu_memory(label: str):
    if not torch.cuda.is_available():
        print(f"{label}: CUDA not available")
        return
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"{label}: {allocated:.2f} GiB allocated, {reserved:.2f} GiB reserved, {total:.2f} GiB total")


def first_model_device(model) -> torch.device:
    for device in getattr(model, "hf_device_map", {}).values():
        if device not in {"cpu", "disk"}:
            return torch.device(device)
    return next(model.parameters()).device


def cuda_compute_dtype():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def quantization_config():
    compute_dtype = cuda_compute_dtype()
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_quant_storage=compute_dtype,
    )


def model_load_kwargs(for_training: bool = False) -> dict:
    if torch.cuda.is_available():
        kwargs = {
            "quantization_config": quantization_config(),
            "dtype": cuda_compute_dtype(),
            "attn_implementation": "eager",
        }
        kwargs["device_map"] = {"": 0} if FORCE_GPU_ONLY_LOAD else "auto"
        return kwargs

    if for_training:
        raise RuntimeError(
            "QLoRA training requires a CUDA GPU. No CUDA GPU was detected."
        )

    return {
        "dtype": torch.float32,
        "low_cpu_mem_usage": True,
        "attn_implementation": "eager",
    }


def load_inference_model(adapter_dir: Path | None = None):
    release_model_memory()
    print_gpu_memory("Before model load")

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
    )
    processor.tokenizer.padding_side = "right"

    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
        **model_load_kwargs(for_training=False),
    )
    if adapter_dir is not None:
        from peft import PeftModel

        model = PeftModel.from_pretrained(model, str(adapter_dir))
    model.eval()
    print_gpu_memory("After model load")
    return model, processor


def blocked_generation_tokens(tokenizer) -> list[list[int]]:
    blocked = []
    for token in ("<unused94>", "thought"):
        token_ids = tokenizer.encode(token, add_special_tokens=False)
        if token_ids:
            blocked.append(token_ids)
    return blocked


def generate_esi(model, tokenizer, user_prompt: str):
    user_prompt = f"{user_prompt}\n\nAnswer with only the value for predicted_esi_level."
    chat_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_prompt}],
        add_generation_prompt=True,
        tokenize=False,
    ) + "predicted_esi_level:"

    inputs = tokenizer(chat_text, return_tensors="pt", return_dict=True)
    device = first_model_device(model)
    inputs = {
        key: value.to(device) if hasattr(value, "to") else value
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            bad_words_ids=blocked_generation_tokens(tokenizer),
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=[
                tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids("<end_of_turn>"),
            ],
            return_dict_in_generate=True,
            output_scores=True,
        )

    new_token_start = inputs["input_ids"].shape[-1]
    generated = tokenizer.decode(
        output.sequences[0][new_token_start:],
        skip_special_tokens=True,
    ).strip()

    confidence = None
    if output.scores:
        first_token_id = int(output.sequences[0][new_token_start].item())
        first_token_scores = output.scores[0][0].float()
        confidence = float(torch.softmax(first_token_scores, dim=-1)[first_token_id].item())

    return parse_esi_prediction(generated), generated, confidence


def evaluate_records(model, tokenizer, records: list[dict], description: str):
    predictions = []
    for record in tqdm(records, desc=description, unit="case"):
        user_prompt = record["messages"][0]["content"]
        prediction, raw_response, confidence = generate_esi(model, tokenizer, user_prompt)
        review, review_reason = needs_human_review(prediction, confidence)
        predictions.append(
            {
                "subject_id": record["subject_id"],
                "stay_id": record["stay_id"],
                "actual": record["label"],
                "prediction": prediction,
                "prediction_confidence": confidence,
                "needs_human_review": review,
                "review_reason": review_reason,
                "raw_response": raw_response,
            }
        )
    return predictions


## Base-model evaluation

In [ ]:
BASE_EVALUATION_PATH = OUTPUT_DIR / "base_evaluation.json"

if RUN_BASE_EVALUATION:
    release_model_memory()
    base_model, base_processor = load_inference_model()
    base_predictions = evaluate_records(
        base_model,
        base_processor.tokenizer,
        test_records,
        "Base MedGemma",
    )
    base_result = save_evaluation("base_evaluation", base_predictions)
    display_metrics(base_result)
    release_model_memory()
elif BASE_EVALUATION_PATH.exists():
    base_result = json.loads(BASE_EVALUATION_PATH.read_text(encoding="utf-8"))
    print("Loaded existing profile-specific base evaluation.")
    display_metrics(base_result)
else:
    print("Set RUN_BASE_EVALUATION = True and rerun this cell.")


## QLoRA training

All profiles train LoRA adapters on the language model while leaving the
vision tower frozen. The `full` profile uses the full prepared split; it is
still parameter-efficient fine-tuning, not full-parameter training.

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


def to_sft_dataset(records: list[dict]) -> Dataset:
    return Dataset.from_list(
        [
            {
                "prompt": [record["messages"][0]],
                "completion": [record["messages"][1]],
            }
            for record in records
        ]
    )


train_dataset = to_sft_dataset(train_records)
validation_dataset = to_sft_dataset(validation_records)

display(train_dataset)
print(f"{len(train_dataset)} training records")

In [ ]:
LANGUAGE_LORA_TARGETS = (
    r".*language_model\.layers\.\d+\."
    r"(?:self_attn\.(?:q_proj|k_proj|v_proj|o_proj)|"
    r"mlp\.(?:gate_proj|up_proj|down_proj))$"
)


def hardware_snapshot() -> dict:
    snapshot = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "platform": platform.platform(),
        "python": sys.version,
        "pytorch": torch.__version__,
        "cuda_build": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
    }
    try:
        import transformers
        import trl
        import peft
        import bitsandbytes

        snapshot["packages"] = {
            "transformers": transformers.__version__,
            "trl": trl.__version__,
            "peft": peft.__version__,
            "bitsandbytes": bitsandbytes.__version__,
        }
    except ImportError:
        pass

    try:
        import psutil

        snapshot["system_ram_gib"] = psutil.virtual_memory().total / 1024**3
    except ImportError:
        snapshot["system_ram_gib"] = None

    if torch.cuda.is_available():
        properties = torch.cuda.get_device_properties(0)
        snapshot["gpu"] = {
            "name": properties.name,
            "total_vram_gib": properties.total_memory / 1024**3,
            "compute_capability": list(torch.cuda.get_device_capability(0)),
            "allocated_gib": torch.cuda.memory_allocated(0) / 1024**3,
            "reserved_gib": torch.cuda.memory_reserved(0) / 1024**3,
            "peak_allocated_gib": torch.cuda.max_memory_allocated(0) / 1024**3,
            "peak_reserved_gib": torch.cuda.max_memory_reserved(0) / 1024**3,
        }
    return snapshot


if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "QLoRA training requires a CUDA GPU. "
            "Run base evaluation on CPU, or use a machine with CUDA for training."
        )

    gpu_properties = torch.cuda.get_device_properties(0)
    total_vram_gb = gpu_properties.total_memory / 1024**3
    print(f"Training on {gpu_properties.name} with {total_vram_gb:.1f} GiB VRAM.")

    release_model_memory()
    torch.cuda.reset_peak_memory_stats()
    training_started = time.perf_counter()

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
    )
    processor.tokenizer.padding_side = "right"

    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        local_files_only=LOCAL_FILES_ONLY,
        **model_load_kwargs(for_training=True),
    )
    model.config.use_cache = False

    peft_config = LoraConfig(
        r=CONFIG["lora_r"],
        lora_alpha=CONFIG["lora_alpha"],
        lora_dropout=0.05,
        bias="none",
        target_modules=LANGUAGE_LORA_TARGETS,
        task_type="CAUSAL_LM",
    )

    training_args = SFTConfig(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=CONFIG["epochs"],
        max_steps=CONFIG["max_steps"],
        per_device_train_batch_size=CONFIG["batch_size"],
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=CONFIG["gradient_accumulation"],
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate=CONFIG["learning_rate"],
        warmup_ratio=0.03,
        max_grad_norm=0.3,
        optim="paged_adamw_8bit",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        max_length=CONFIG["max_length"],
        completion_only_loss=True,
        eval_strategy=CONFIG["eval_strategy"],
        save_strategy=CONFIG["save_strategy"],
        save_steps=CONFIG["save_steps"],
        logging_steps=1,
        logging_first_step=True,
        report_to="none",
        seed=42,
        dataset_num_proc=1,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=(
            validation_dataset
            if CONFIG["eval_strategy"] != "no"
            else None
        ),
        peft_config=peft_config,
        processing_class=processor.tokenizer,
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in trainer.model.parameters()
        if parameter.requires_grad
    )
    vision_trainable = any(
        "vision_tower" in name and parameter.requires_grad
        for name, parameter in trainer.model.named_parameters()
    )
    assert not vision_trainable
    print(f"Trainable parameters: {trainable_parameters:,}")
    print("Vision tower trainable:", vision_trainable)

    train_output = trainer.train()
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(ADAPTER_DIR))
    processor.tokenizer.save_pretrained(ADAPTER_DIR)

    elapsed_seconds = time.perf_counter() - training_started
    training_report = {
        **train_output.metrics,
        "profile": PROFILE,
        "model_id": MODEL_ID,
        "train_records": len(train_records),
        "validation_records": len(validation_records),
        "trainable_parameters": trainable_parameters,
        "elapsed_seconds": elapsed_seconds,
        "config": CONFIG,
        "hardware": hardware_snapshot(),
    }
    (OUTPUT_DIR / "training_metrics.json").write_text(
        json.dumps(training_report, indent=2),
        encoding="utf-8",
    )
    print(f"Adapter saved to {ADAPTER_DIR}")

    release_model_memory()
else:
    print("Set RUN_TRAINING = True and rerun this cell.")

## Adapter evaluation

In [ ]:
ADAPTER_EVALUATION_PATH = OUTPUT_DIR / "adapter_evaluation.json"

if RUN_ADAPTER_EVALUATION:
    if not ADAPTER_DIR.exists():
        raise FileNotFoundError(f"No adapter found at {ADAPTER_DIR}")

    release_model_memory()
    adapter_model, adapter_processor = load_inference_model(ADAPTER_DIR)
    adapter_predictions = evaluate_records(
        adapter_model,
        adapter_processor.tokenizer,
        test_records,
        "Fine-tuned MedGemma",
    )
    adapter_result = save_evaluation("adapter_evaluation", adapter_predictions)
    display_metrics(adapter_result)
    release_model_memory()
elif ADAPTER_EVALUATION_PATH.exists():
    adapter_result = json.loads(ADAPTER_EVALUATION_PATH.read_text(encoding="utf-8"))
    print("Loaded existing profile-specific adapter evaluation.")
    display_metrics(adapter_result)
else:
    print("Train an adapter, then set RUN_ADAPTER_EVALUATION = True.")


## Base and adapter comparison

In [ ]:
def load_result(path: Path):
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


base_result = load_result(BASE_EVALUATION_PATH)
adapter_result = load_result(ADAPTER_EVALUATION_PATH)

if base_result and adapter_result:
    if base_result["test_fingerprint"] != adapter_result["test_fingerprint"]:
        raise ValueError("Base and adapter evaluations used different test records.")

    metric_names = [
        "accuracy",
        "macro_f1",
        "coverage",
        "under_triage_rate",
        "over_triage_rate",
        "severe_under_triage_rate",
        "uncertainty_rate",
        "invalid_output_rate",
        "human_review_rate",
    ]
    comparison = pd.DataFrame(
        {
            "metric": metric_names,
            "base": [base_result["metrics"][name] for name in metric_names],
            "fine_tuned": [adapter_result["metrics"][name] for name in metric_names],
        }
    )
    comparison["change"] = comparison["fine_tuned"] - comparison["base"]
    display(comparison)

    recall_comparison = pd.DataFrame(
        {
            "base_recall": base_result["metrics"]["recall_by_esi"],
            "fine_tuned_recall": adapter_result["metrics"]["recall_by_esi"],
        }
    )
    display(recall_comparison)
else:
    print("Run both evaluations to create a fair comparison.")

## Error review

Invalid output, explicit uncertainty, or low first-token probability is marked
for review. First-token probability is not calibrated clinical confidence.

In [ ]:
def classify_error(row):
    actual = row.get("actual")
    prediction = row.get("prediction")
    if not isinstance(prediction, int):
        return "invalid_or_uncertain"
    if prediction == actual:
        return "correct"
    if actual in (1, 2) and prediction in (4, 5):
        return "severe_under_triage"
    if prediction > actual:
        return "under_triage"
    return "over_triage"


def make_error_table(result: dict) -> pd.DataFrame:
    rows = result.get("predictions", [])
    if not rows:
        return pd.DataFrame()

    table = pd.DataFrame(rows)
    table["correct"] = table["prediction"] == table["actual"]
    table["absolute_error"] = table.apply(
        lambda row: abs(row["prediction"] - row["actual"])
        if isinstance(row["prediction"], int)
        else None,
        axis=1,
    )
    table["error_type"] = table.apply(classify_error, axis=1)
    table["clinically_serious"] = table["error_type"].isin(
        ["severe_under_triage", "under_triage"]
    )

    preferred_columns = [
        "subject_id",
        "stay_id",
        "actual",
        "prediction",
        "correct",
        "error_type",
        "absolute_error",
        "clinically_serious",
        "prediction_confidence",
        "needs_human_review",
        "review_reason",
        "raw_response",
    ]
    columns = [column for column in preferred_columns if column in table.columns]
    return table[columns].sort_values(
        by=["correct", "clinically_serious", "absolute_error"],
        ascending=[True, False, False],
        na_position="last",
    )


def display_error_analysis(name: str, result: dict):
    table = make_error_table(result)
    if table.empty:
        print(f"No {name} predictions found.")
        return

    print(f"{name}: {int((~table['correct']).sum())} errors out of {len(table)} cases")
    display(
        table[~table["correct"]]
        .drop(columns=["subject_id", "stay_id"], errors="ignore")
        .head(30)
    )

    if "needs_human_review" in table.columns:
        review_summary = (
            table.fillna({"review_reason": ""})
            .groupby(["needs_human_review", "review_reason"], dropna=False)
            .size()
            .reset_index(name="cases")
        )
        display(review_summary)


if base_result:
    display_error_analysis("Base model", base_result)

if adapter_result:
    display_error_analysis("Fine-tuned adapter", adapter_result)


## Environment

In [ ]:
environment_report = {
    "profile": PROFILE,
    "model_id": MODEL_ID,
    "config": CONFIG,
    "selected_data": {
        "train_rows": len(train_records),
        "validation_rows": len(validation_records),
        "test_rows": len(test_records),
        "test_fingerprint": test_fingerprint(test_records),
    },
    "hardware": hardware_snapshot(),
}

hardware_path = OUTPUT_DIR / "hardware_environment.json"
hardware_path.write_text(json.dumps(environment_report, indent=2), encoding="utf-8")

display(pd.json_normalize(environment_report))
print(f"Hardware report written to {hardware_path}")

## Notes

- Keep MIMIC-IV-ED data and outputs local.
- Missing triage values are preserved as `null`.
- Severe under-triage matters more than aggregate accuracy.
- Results from this notebook are not clinical validation.